In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

In [3]:
import logging
logger = logging.getLogger()
logger.setLevel(logging.INFO)

In [4]:
import json
import os
import re
import csv
import pandas as pd
from tqdm import tqdm
import gzip

In [5]:
def process_scrutin(scrutin: dict = None):

    all_votants = []

    main_data_keys = [
        "uid",
        "numero",
        "organeRef",
        "legislature",
        "seanceRef",
        "dateScrutin",
        "quantiemeJourSeance",
        "typeVote/codeTypeVote",
        "typeVote/typeMajorité",
        "sort/code",
        "titre",
        "objet/libelle",
        "syntheseVote/nombreVotants",
        "syntheseVote/suffragesExprimes",
        "syntheseVote/nbrSuffragesRequis",
        "syntheseVote/annonce",
        "syntheseVote/decompte/nonVotants",
        "syntheseVote/decompte/pour",
        "syntheseVote/decompte/contre",
        "syntheseVote/decompte/abstentions",
        "syntheseVote/decompte/nonVotantsVolontaires",
    ]

    main_data = {}

    for item in main_data_keys:
        parts = item.split("/")
        init_part = scrutin.get(parts[0])
        for key in parts[1:]:
            init_part = init_part.get(key)
        main_data[f"main/{item}"] = init_part

    for _, institution in scrutin.get("ventilationVotes").items():
        institution_organeRef = institution.get("organeRef")

        for group in institution.get("groupes").get("groupe"):
            data_group_keys = [
                "organeRef",
                "nombreMembresGroupe",
                "vote/positionMajoritaire",
                "vote/decompteVoix/nonVotants",
                "vote/decompteVoix/pour",
                "vote/decompteVoix/contre",
                "vote/decompteVoix/abstentions",
                "vote/decompteVoix/nonVotantsVolontaires"
            ]

            group_data = {}

            for item in data_group_keys:
                parts = item.split("/")
                init_part = group.get(parts[0])
                for key in parts[1:]:
                    init_part = init_part.get(key)
                group_data[f"groupe/{item}"] = init_part



            for votant_type, votant_dict in group.get("vote").get("decompteNominatif").items():
                if votant_dict is None:
                    continue

                if votant_dict == "0":
                    continue

                # print(votant_type)

                votant_list = votant_dict.get("votant")

                if type(votant_list) == dict:
                    votant_payload = {}
                    votant_payload.update({f"votant/{k}": str(v) for k, v in votant_list.items()})
                    votant_payload.update({k: str(v) for k, v in group_data.items()})
                    votant_payload.update({k: str(v) for k, v in main_data.items()})
                    votant_payload.update({"votant/vote": votant_type})
                    all_votants.append(votant_payload)
                    continue

                for votant in votant_list:
                    votant_payload = {}
                    votant_payload.update({f"votant/{k}": str(v) for k, v in votant.items()})
                    votant_payload.update({k: str(v) for k, v in group_data.items()})
                    votant_payload.update({k: str(v) for k, v in main_data.items()})
                    votant_payload.update({"votant/vote": votant_type})
                    all_votants.append(votant_payload)

    return all_votants

## XV Législature

In [ ]:
# Cell: Unzip Scrutins_XV.json.zip
import zipfile

# Define paths
here = os.getcwd()
folder = os.path.dirname(here)
folder_0 = folder + "/0_raw"
zip_path = os.path.join(folder_0, "Scrutins_XV.json.zip")  # ← Fixed filename
extract_path = os.path.join(folder_0, "json")

# Create extraction directory if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Unzip the file
print(f"Unzipping {zip_path}...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(folder_0)  # ← Extract to folder_0, not extract_path
    
print(f"Extracted to: {folder_0}")

# Check what was extracted
json_dir = os.path.join(folder_0, "json")
if os.path.exists(json_dir):
    json_files = [f for f in os.listdir(json_dir) if f.endswith('.json')]
    print(f"Found {len(json_files)} JSON files in: {json_dir}")
else:
    print(f"ERROR: 'json' folder not found in {folder_0}")

Unzipping c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN\Scrutins\Scrutins\1_data_extraction\15e_legislature_27juin2017-21juin2022/0_raw\Scrutins_XV.json.zip...
Extracted to: c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN\Scrutins\Scrutins\1_data_extraction\15e_legislature_27juin2017-21juin2022/0_raw
Found 4417 JSON files in: c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN\Scrutins\Scrutins\1_data_extraction\15e_legislature_27juin2017-21juin2022/0_raw\json


In [9]:
# Cell: Load and process JSON files
database = []

# Point to the extracted JSON directory
data_dir = os.path.join(folder_0, "json")

for filename in tqdm(os.listdir(data_dir)):
    if not re.match("^.*\.json$", filename):
        continue

    source_filename = os.path.join(data_dir, filename)

    with open(source_filename, "r", encoding="UTF-8") as input_file:
        content = json.load(input_file)

    all_votants = process_scrutin(scrutin=content.get("scrutin"))
    database.extend(all_votants)

df = pd.DataFrame.from_records(database)

# Save to CSV only
output_path = os.path.join(folder, "1_extract_python/scrutins.csv")
df.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Dataset shape: {df.shape}")

100%|██████████| 4417/4417 [01:41<00:00, 43.44it/s]


Saved to: c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\AN\Scrutins\Scrutins\1_data_extraction\15e_legislature_27juin2017-21juin2022\1_extract_python/scrutins.csv
Dataset shape: (472631, 34)


In [12]:
df

,votant/acteurRef,votant/mandatRef,votant/causePositionVote,groupe/organeRef,groupe/nombreMembresGroupe,groupe/vote/positionMajoritaire,groupe/vote/decompteVoix/nonVotants,groupe/vote/decompteVoix/pour,groupe/vote/decompteVoix/contre,groupe/vote/decompteVoix/abstentions,...,main/syntheseVote/suffragesExprimes,main/syntheseVote/nbrSuffragesRequis,main/syntheseVote/annonce,main/syntheseVote/decompte/nonVotants,main/syntheseVote/decompte/pour,main/syntheseVote/decompte/contre,main/syntheseVote/decompte/abstentions,main/syntheseVote/decompte/nonVotantsVolontaires,votant/vote,votant/parDelegation
0,PA606171,PM722632,PAN,PO730964,271,contre,1,1,29,0,...,49,25,L'Assemblée nationale n'a pas adopté,2,17,32,1,0,nonVotants,NaN
1,PA719350,PM722634,NaN,PO730964,271,contre,1,1,29,0,...,49,25,L'Assemblée nationale n'a pas adopté,2,17,32,1,0,pours,false
2,PA2449,PM723036,NaN,PO730964,271,contre,1,1,29,0,...,49,25,L'Assemblée nationale n'a pas adopté,2,17,32,1,0,contres,false
3,PA719952,PM722814,NaN,PO730964,271,contre,1,1,29,0,...,49,25,L'Assemblée nationale n'a pas adopté,2,17,32,1,0,contres,false
4,PA672,PM723230,NaN,PO730964,271,contre,1,1,29,0,...,49,25,L'Assemblée nationale n'a pas adopté,2,17,32,1,0,contres,false
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
472626,PA720546,PM722996,NaN,PO730940,16,contre,0,0,9,0,...,206,104,l'Assemblée nationale a adopté,2,169,37,3,0,contres,false
472627,PA721118,PM723176,NaN,PO730940,16,contre,0,0,9,0,...,206,104,l'Assemblée nationale a adopté,2,169,37,3,0,contres,false
472628,PA721896,PM723414,NaN,PO730940,16,contre,0,0,9,0,...,206,104,l'Assemblée nationale a adopté,2,169,37,3,0,contres,false
472629,PA722110,PM723474,NaN,PO730940,16,contre,0,0,9,0,...,206,104,l'Assemblée nationale a adopté,2,169,37,3,0,contres,false
